# 🚀 Scientific Python Workflow on HPC: From Local Prototyping to Cluster & GPU Acceleration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/erolrecep/scientific_python_workflow_hpc/blob/main/notebooks/scientific_python_workflow_colab.ipynb)

**Instructor:** Erol Recep  
**GitHub Repository:** [scientific_python_workflow_hpc](https://github.com/erolrecep/scientific_python_workflow_hpc)  
**Workshop Topic:** Production Scientific Python, DAG Workflows, Multi-Core Scaling, and GPU Batch Acceleration.  

---

### 🎯 Colab Setup Instructions (Enable Free GPU):
1. In the top menu bar, click **Runtime** ➔ **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** (or any available GPU).
3. Click **Save**.
4. Run the cells below sequentially by pressing `Shift + Enter`.

## 1. 🔍 Environment & Hardware Discovery
Let's inspect the allocated Google Colab compute node (CPU cores, RAM, and NVIDIA GPU).

In [ ]:
import os
import platform
import torch

print(f"OS Platform:       {platform.platform()}")
print(f"Colab vCPUs:       {os.cpu_count()}")

# Check NVIDIA GPU
has_gpu = torch.cuda.is_available()
print(f"CUDA Available:    {has_gpu}")
if has_gpu:
    print(f"GPU Name:          {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory:        {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    !nvidia-smi
else:
    print("⚠️ No GPU detected. If you want GPU acceleration, switch runtime to T4 GPU via Runtime > Change runtime type.")

## 2. 📦 Clone Repository & Bootstrap `uv`
We clone the official workshop repository and install all dependencies using `uv`.

In [ ]:
# Clone the repository if running inside Colab fresh environment
if not os.path.exists("src"):
    if not os.path.exists("scientific_python_workflow_hpc"):
        !git clone https://github.com/erolrecep/scientific_python_workflow_hpc.git
    %cd scientific_python_workflow_hpc

# Bootstrap standalone uv package manager
!curl -LsSf https://astral.sh/uv/install.sh | sh
uv_bin = os.path.expanduser("~/.local/bin")
if uv_bin not in os.environ["PATH"]:
    os.environ["PATH"] = f"{uv_bin}:" + os.environ["PATH"]

!uv --version

# Install requirements into Colab environment
!uv pip install --system -r requirements.txt

## 3. 📥 Download & Verify Dataset
We download the Oxford 17flowers benchmark dataset (~1,360 images).

In [ ]:
!python src/download_data.py

import glob
images = glob.glob("data/jpg/*.jpg")
print(f"✅ Successfully verified dataset: {len(images)} images ready in data/jpg/")

## 4. 🧪 Run Test Suite
Run automated tests to ensure pipeline integrity, image dimension compliance (229x229), and mathematical equivalence across all backends (OpenCV, Pillow, PyTorch CPU, and CUDA GPU).

In [ ]:
!pytest tests/ -v

## 5. ⚡ Part 1: Single-Core Sequential Baseline
We establish our 1-core baseline performance across OpenCV and Pillow.

In [ ]:
print("=== OpenCV Single-Core Baseline ===")
!python src/process_images.py --benchmark --backend opencv --workers 1 --max-images 300

print("\n=== Pillow Single-Core Baseline ===")
!python src/process_images.py --benchmark --backend pillow --workers 1 --max-images 300

## 6. 🚀 Part 2: Multi-Core Parallel Processing
Using Python's `ProcessPoolExecutor` with anti-oversubscription thread pinning (`cv2.setNumThreads(1)`), we scale processing across all allocated Colab vCPUs.

In [ ]:
workers = os.cpu_count() or 2
print(f"Benchmarking Multi-Core Scaling using {workers} Workers...")
!python src/process_images.py --benchmark --backend opencv --workers {workers} --max-images 500

## 7. ⚡ Part 3: Hardware-Accelerated GPU Batch Processing (CUDA)
Offload image processing to the GPU: pipelined host-to-device streaming, CUDA 2D convolution sharpening filters (`F.conv2d`), and GPU bilinear resizing (`F.interpolate`).

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running GPU Batch Pipeline on: {device.upper()}")
!python src/process_images.py --benchmark --backend gpu --device {device} --batch-size 64 --max-images 500

## 8. 📊 Part 4: Back-to-Back Comparison Across All Backends
We now execute our comprehensive benchmark comparing:
1. **OpenCV** (Single-core & Multi-core)
2. **Pillow** (Single-core & Multi-core)
3. **PyTorch CPU** (Single-core & Multi-core)
4. **PyTorch CUDA GPU** (Batch Hardware Acceleration)

In [ ]:
!mkdir -p results
!python src/process_images.py --benchmark --backend all --workers {workers} --device {device} --batch-size 64 --max-images 500 | tee results/colab_benchmark.txt

## 9. 📈 Interactive Visualization
Plot the throughput (images per second) and speedup comparison directly inside Google Colab.

In [ ]:
import re
import matplotlib.pyplot as plt

# Parse the benchmark results file
data = {}
if os.path.exists("results/colab_benchmark.txt"):
    with open("results/colab_benchmark.txt") as f:
        for line in f:
            # Match lines like: | opencv               | 500       | 16       | 0.34s     | 1462.7 img/s |
            match = re.search(r"\|\s+([a-zA-Z0-9_ -]+?)\s+\|\s+\d+\s+\|\s+([a-zA-Z0-9]+)\s+\|\s+([0-9.]+)s\s+\|\s+([0-9.]+)\s+img/s", line)
            if match:
                name, workers_str, time_s, throughput = match.groups()
                label = f"{name.strip()} ({workers_str.strip()}w)"
                data[label] = float(throughput)

if data:
    labels = list(data.keys())
    values = list(data.values())
    colors = ["#3498db" if "1w" in l else "#2ecc71" if "gpu" not in l.lower() else "#e74c3c" for l in labels]

    plt.figure(figsize=(10, 5), dpi=100)
    bars = plt.barh(labels, values, color=colors)
    plt.xlabel("Throughput (Images / Second)", fontsize=12, fontweight="bold")
    plt.title("Scientific Image Processing: HPC Scaling Benchmark", fontsize=14, fontweight="bold", pad=15)
    plt.grid(axis="x", linestyle="--", alpha=0.6)

    for bar in bars:
        width = bar.get_width()
        plt.text(width + 10, bar.get_y() + bar.get_height()/2, f"{width:.1f} img/s",
                 va="center", ha="left", fontsize=10, fontweight="bold")

    plt.tight_layout()
    plt.show()
else:
    print("Run cell 8 first to generate results/colab_benchmark.txt!")

## 10. 🎓 Key Takeaways for HPC Clusters

1. **Zero-Configuration Entry**: Google Colab gives any workshop attendee immediate access to Linux and NVIDIA GPUs without installing software locally.
2. **Direct Parity with Supercomputers**: The exact code executed here runs identically on Rocky Linux SLURM supercomputers via `Makefile.sbatch` and `sbatch/*.sbatch` scripts.
3. **Multi-Paradigm Scaling**: When scaling scientific workloads:
   - Multi-core CPU (`ProcessPoolExecutor`) provides linear speedup for CPU-bound tasks.
   - CUDA GPU streaming unlocks 5-10x throughput for batch tensor operations.